# Colab to Drive

In [0]:
# Install a Drive FUSE wrapper.
# https://github.com/astrada/google-drive-ocamlfuse
!apt-get install -y -qq software-properties-common python-software-properties module-init-tools
!add-apt-repository -y ppa:alessandro-strada/ppa 2>&1 > /dev/null
!apt-get update -qq 2>&1 > /dev/null
!apt-get -y install -qq google-drive-ocamlfuse fuse

E: Package 'python-software-properties' has no installation candidate


In [0]:
# Generate auth tokens for Colab
from google.colab import auth
auth.authenticate_user()

In [0]:
# Generate creds for the Drive FUSE library.
from oauth2client.client import GoogleCredentials
creds = GoogleCredentials.get_application_default()
import getpass
!google-drive-ocamlfuse -headless -id={creds.client_id} -secret={creds.client_secret} < /dev/null 2>&1 | grep URL
vcode = getpass.getpass()
!echo {vcode} | google-drive-ocamlfuse -headless -id={creds.client_id} -secret={creds.client_secret}

In [0]:
# Create a directory and mount Google Drive using that directory.
!mkdir -p drive!
!google-drive-ocamlfuse drive

# Tensorflow Object Detection API example

In [0]:
import os as os
os.chdir('/content')

!wget --header="Host: doc-14-9c-docs.googleusercontent.com" --header="User-Agent: Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/75.0.3770.100 Safari/537.36" --header="Accept: text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3" --header="Accept-Language: en-US,en;q=0.9" --header="Referer: https://drive.google.com/drive/u/0/my-drive" --header="Cookie: AUTH_4lgch8kproorp87c5erpsi9rqpbk46dg=14696292948499217652|1561564800000|07bh47pi86cqaku3giic6thsg3l3qije" --header="Connection: keep-alive" "https://doc-14-9c-docs.googleusercontent.com/docs/securesc/0ga3her86at4fpdr2b9n3spfq1b4g359/nns4meldejs7e2sc42gug952pnosjt3b/1561564800000/14696292948499217652/14696292948499217652/1oWjVHVwJMfCoRWsmyfF1cS01D1Rizkl5?h=06864318269717761326&e=download" -O "IITD_Research_Intern.zip" -c

--2019-06-26 16:44:47--  https://doc-14-9c-docs.googleusercontent.com/docs/securesc/0ga3her86at4fpdr2b9n3spfq1b4g359/nns4meldejs7e2sc42gug952pnosjt3b/1561564800000/14696292948499217652/14696292948499217652/1oWjVHVwJMfCoRWsmyfF1cS01D1Rizkl5?h=06864318269717761326&e=download
Resolving doc-14-9c-docs.googleusercontent.com (doc-14-9c-docs.googleusercontent.com)... 74.125.142.132, 2607:f8b0:400e:c08::84
Connecting to doc-14-9c-docs.googleusercontent.com (doc-14-9c-docs.googleusercontent.com)|74.125.142.132|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [application/x-zip-compressed]
Saving to: ‘IITD_Research_Intern.zip’

IITD_Research_Inter     [             <=>    ] 768.57M   140MB/s    in 6.1s    

2019-06-26 16:44:53 (126 MB/s) - ‘IITD_Research_Intern.zip’ saved [805908158]



In [0]:
!unzip 'IITD_Research_Intern.zip'

Archive:  IITD_Research_Intern.zip
   creating: IITD_Research_Intern/
  inflating: IITD_Research_Intern/Faster_RCNN.pdf  
   creating: IITD_Research_Intern/models/
  inflating: IITD_Research_Intern/models/.gitignore  
  inflating: IITD_Research_Intern/models/.gitmodules  
  inflating: IITD_Research_Intern/models/AUTHORS  
  inflating: IITD_Research_Intern/models/CODEOWNERS  
  inflating: IITD_Research_Intern/models/CONTRIBUTING.md  
  inflating: IITD_Research_Intern/models/ISSUE_TEMPLATE.md  
  inflating: IITD_Research_Intern/models/LICENSE  
   creating: IITD_Research_Intern/models/official/
  inflating: IITD_Research_Intern/models/official/.gitignore  
   creating: IITD_Research_Intern/models/official/benchmark/
  inflating: IITD_Research_Intern/models/official/benchmark/benchmark_uploader.py  
  inflating: IITD_Research_Intern/models/official/benchmark/benchmark_uploader_main.py  
  inflating: IITD_Research_Intern/models/official/benchmark/benchmark_uploader_test.py  
   creating: I

In [0]:
!unzip 'models.zip'

In [0]:
import os as os
os.chdir('/content/models/research/object_detection')

In [0]:
import numpy as np
import os
import six.moves.urllib as urllib
import sys
import tarfile
import tensorflow as tf
import zipfile

from distutils.version import StrictVersion
from collections import defaultdict
from io import StringIO
from matplotlib import pyplot as plt
from PIL import Image

# This is needed since the notebook is stored in the object_detection folder.
sys.path.append("..")
from object_detection.utils import ops as utils_ops

# if StrictVersion(tf.__version__) < StrictVersion('1.12.0'):
#   raise ImportError('Please upgrade your TensorFlow installation to v1.12.*.')

In [0]:
import os as os
os.chdir('/content/models/research/object_detection/')

In [0]:
# This is needed to display the images.
%matplotlib inline

In [0]:
from utils import label_map_util

from utils import visualization_utils as vis_util

In [0]:
!wget --header="Host: download.tensorflow.org" --header="User-Agent: Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/74.0.3729.169 Safari/537.36" --header="Accept: text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3" --header="Accept-Language: en-US,en;q=0.9" "http://download.tensorflow.org/models/object_detection/faster_rcnn_resnet50_coco_2018_01_28.tar.gz" -O "faster_rcnn_resnet50_coco_2018_01_28.tar.gz" -c
!tar -xzvf faster_rcnn_resnet50_coco_2018_01_28.tar.gz

In [0]:
# What model to download.
MODEL_NAME = 'ssd_mobilenet_v1_coco_2018_01_28'

# Path to frozen detection graph. This is the actual model that is used for the object detection.
PATH_TO_FROZEN_GRAPH = MODEL_NAME + '/frozen_inference_graph.pb'

# List of the strings that is used to add correct label for each box.
PATH_TO_LABELS = os.path.join('data', 'mscoco_label_map.pbtxt')

In [0]:
detection_graph = tf.Graph()
with detection_graph.as_default():
  od_graph_def = tf.GraphDef()
  with tf.gfile.GFile(PATH_TO_FROZEN_GRAPH, 'rb') as fid:
    serialized_graph = fid.read()
    od_graph_def.ParseFromString(serialized_graph)
    tf.import_graph_def(od_graph_def, name='')

In [0]:
category_index = label_map_util.create_category_index_from_labelmap(PATH_TO_LABELS, use_display_name=True)

W0626 16:13:27.999326 140307363571584 deprecation_wrapper.py:119] From /content/models/research/object_detection/utils/label_map_util.py:132: The name tf.gfile.GFile is deprecated. Please use tf.io.gfile.GFile instead.



In [0]:
def load_image_into_numpy_array(image):
  (im_width, im_height) = image.size
  return np.array(image.getdata()).reshape(
      (im_height, im_width, 3)).astype(np.uint8)

In [0]:
# For the sake of simplicity we will use only 2 images:
# image1.jpg
# image2.jpg
# If you want to test the code with your images, just add path to the images to the TEST_IMAGE_PATHS.
PATH_TO_TEST_IMAGES_DIR = 'test_images'
TEST_IMAGE_PATHS = [ os.path.join(PATH_TO_TEST_IMAGES_DIR, 'cow{}.jpg'.format(i)) for i in range(1000, 6360, 11) ]

# Size, in inches, of the output images.
IMAGE_SIZE = (12, 8)
file = open('geek.txt', 'w')

In [0]:
def run_inference_for_single_image(image, graph):
  with graph.as_default():
    with tf.Session() as sess:
      # Get handles to input and output tensors
      ops = tf.get_default_graph().get_operations()
      all_tensor_names = {output.name for op in ops for output in op.outputs}
      tensor_dict = {}
      for key in [
          'num_detections', 'detection_boxes', 'detection_scores',
          'detection_classes', 'detection_masks'
      ]:
        tensor_name = key + ':0'
        if tensor_name in all_tensor_names:
          tensor_dict[key] = tf.get_default_graph().get_tensor_by_name(
              tensor_name)
      if 'detection_masks' in tensor_dict:
        # The following processing is only for single image
        detection_boxes = tf.squeeze(tensor_dict['detection_boxes'], [0])
        detection_masks = tf.squeeze(tensor_dict['detection_masks'], [0])
        # Reframe is required to translate mask from box coordinates to image coordinates and fit the image size.
        real_num_detection = tf.cast(tensor_dict['num_detections'][0], tf.int32)
        detection_boxes = tf.slice(detection_boxes, [0, 0], [real_num_detection, -1])
        detection_masks = tf.slice(detection_masks, [0, 0, 0], [real_num_detection, -1, -1])
        detection_masks_reframed = utils_ops.reframe_box_masks_to_image_masks(
            detection_masks, detection_boxes, image.shape[1], image.shape[2])
        detection_masks_reframed = tf.cast(
            tf.greater(detection_masks_reframed, 0.5), tf.uint8)
        # Follow the convention by adding back the batch dimension
        tensor_dict['detection_masks'] = tf.expand_dims(
            detection_masks_reframed, 0)
      image_tensor = tf.get_default_graph().get_tensor_by_name('image_tensor:0')

      # Run inference
      output_dict = sess.run(tensor_dict,
                             feed_dict={image_tensor: image})

      # all outputs are float32 numpy arrays, so convert types as appropriate
      output_dict['num_detections'] = int(output_dict['num_detections'][0])
      output_dict['detection_classes'] = output_dict[
          'detection_classes'][0].astype(np.int64)
      output_dict['detection_boxes'] = output_dict['detection_boxes'][0]
      output_dict['detection_scores'] = output_dict['detection_scores'][0]
      if 'detection_masks' in output_dict:
        output_dict['detection_masks'] = output_dict['detection_masks'][0]
  return output_dict

In [0]:
from tqdm import tqdm
for image_path in tqdm(TEST_IMAGE_PATHS):
  image = Image.open(image_path)
  # the array based representation of the image will be used later in order to prepare the
  # result image with boxes and labels on it.
  image_np = load_image_into_numpy_array(image)
  # Expand dimensions since the model expects images to have shape: [1, None, None, 3]
  image_np_expanded = np.expand_dims(image_np, axis=0)
  # Actual detection.
  output_dict = run_inference_for_single_image(image_np_expanded, detection_graph)
  # Visualization of the results of a detection.
  vis_util.visualize_boxes_and_labels_on_image_array(
      image_np,
      output_dict['detection_boxes'],
      output_dict['detection_classes'],
      output_dict['detection_scores'],
      category_index,
      instance_masks=output_dict.get('detection_masks'),
      use_normalized_coordinates=True,
      line_thickness=8)
  plt.figure(figsize=IMAGE_SIZE)
  plt.imsave('frcnn_pretrained_results/'+image_path[12:], image_np)
  file.write(image_path,output_dict['detection_boxes'],category_index)
  #plt.imshow(image_np)
file.close()

In [0]:
!zip -r 'frcnn_pretrained_results.zip' frcnn_pretrained_results

# Training On Custom Images

In [0]:
import os as os
os.chdir('/content/IITD_Research_Intern/object-detection/models/research/')
!protoc object_detection/protos/*.proto --python_out=.

In [0]:
os.chdir('/content/IITD_Research_Intern/object-detection/object_detection/models/research')
!sudo python3 setup.py install

FileNotFoundError: ignored

In [0]:
!pwd

/content/models/research


In [0]:
import os as os
os.chdir('/content/IITD_Research_Intern/object-detection/')

In [0]:
!python3 'xml_to_csv.py'

Successfully converted xml to csv.
Successfully converted xml to csv.


In [0]:
!wget --header="Host: doc-0s-9c-docs.googleusercontent.com" --header="User-Agent: Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/75.0.3770.100 Safari/537.36" --header="Accept: text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3" --header="Accept-Language: en-US,en;q=0.9" --header="Referer: https://drive.google.com/drive/u/0/my-drive" --header="Cookie: AUTH_4lgch8kproorp87c5erpsi9rqpbk46dg=14696292948499217652|1561564800000|07bh47pi86cqaku3giic6thsg3l3qije" --header="Connection: keep-alive" "https://doc-0s-9c-docs.googleusercontent.com/docs/securesc/0ga3her86at4fpdr2b9n3spfq1b4g359/ocdlenqimh7rgl4pijlk4pti1lv9sl2r/1561564800000/14696292948499217652/14696292948499217652/1ebVv554iuhb2n9GZWqohSXmIJJ7lerjP?h=06864318269717761326&e=download" -O "images.zip" -c
!unzip 'images.zip'

--2019-06-26 17:02:24--  https://doc-0s-9c-docs.googleusercontent.com/docs/securesc/0ga3her86at4fpdr2b9n3spfq1b4g359/ocdlenqimh7rgl4pijlk4pti1lv9sl2r/1561564800000/14696292948499217652/14696292948499217652/1ebVv554iuhb2n9GZWqohSXmIJJ7lerjP?h=06864318269717761326&e=download
Resolving doc-0s-9c-docs.googleusercontent.com (doc-0s-9c-docs.googleusercontent.com)... 74.125.142.132, 2607:f8b0:400e:c08::84
Connecting to doc-0s-9c-docs.googleusercontent.com (doc-0s-9c-docs.googleusercontent.com)|74.125.142.132|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [application/x-zip-compressed]
Saving to: ‘images.zip’

images.zip              [           <=>      ] 281.31M  70.3MB/s    in 4.0s    

2019-06-26 17:02:29 (70.3 MB/s) - ‘images.zip’ saved [294974693]

Archive:  images.zip
   creating: images/
   creating: images/test/
  inflating: images/test/1.jpg       
  inflating: images/test/1.xml       
   creating: images/train/
  inflating: images/train/2.jpeg 

In [0]:
!python3 generate_tf_record.py --csv_input=data/train_labels.csv --image_dir=images/train --output_path=data/train.record

W0626 17:02:50.866745 140327641307008 deprecation_wrapper.py:119] From generate_tf_record.py:100: The name tf.app.run is deprecated. Please use tf.compat.v1.app.run instead.

W0626 17:02:50.867315 140327641307008 deprecation_wrapper.py:119] From generate_tf_record.py:86: The name tf.python_io.TFRecordWriter is deprecated. Please use tf.io.TFRecordWriter instead.

W0626 17:02:51.815319 140327641307008 deprecation_wrapper.py:119] From generate_tf_record.py:45: The name tf.gfile.GFile is deprecated. Please use tf.io.gfile.GFile instead.

Successfully created the TFRecords: /content/IITD_Research_Intern/object-detection/data/train.record


In [0]:
!python3 generate_tf_record.py --csv_input=data/test_labels.csv --image_dir=images/test --output_path=data/test.record

W0626 17:03:04.076368 140571374692224 deprecation_wrapper.py:119] From generate_tf_record.py:100: The name tf.app.run is deprecated. Please use tf.compat.v1.app.run instead.

W0626 17:03:04.076933 140571374692224 deprecation_wrapper.py:119] From generate_tf_record.py:86: The name tf.python_io.TFRecordWriter is deprecated. Please use tf.io.TFRecordWriter instead.

W0626 17:03:04.190825 140571374692224 deprecation_wrapper.py:119] From generate_tf_record.py:45: The name tf.gfile.GFile is deprecated. Please use tf.io.gfile.GFile instead.

Successfully created the TFRecords: /content/IITD_Research_Intern/object-detection/data/test.record


In [0]:
!wget https://github.com/tensorflow/models/blob/master/research/object_detection/samples/configs/ssd_mobilenet_v1_coco.config

In [0]:
!wget http://download.tensorflow.org/models/object_detection/ssd_mobilenet_v1_coco_2018_01_28.tar.gz

In [0]:
!tar xvzf 'ssd_mobilenet_v1_coco_2018_01_28.tar.gz'

In [0]:
!zip -r IITD_Research_Intern.zip 'IITD_Research_Intern'

In [0]:
os.chdir('/content/IITD_Research_Intern/models/research/object_detection/')

In [0]:
!python3 train.py --logtostderr --train_dir=training/ --pipeline_config_path=training/ssd_mobilenet_v1_coco.config

In [0]:
os.chdir('/content/IITD_Research_Intern/models/research/')

In [0]:
!git clone https://github.com/cocodataset/cocoapi.git
!cd cocoapi/PythonAPI; make; cp -r pycocotools /content/models/research/

In [0]:
!cd /content/IITD_Research_Intern/models/research
!mkdir train eval
%set_env PYTHONPATH=/content/IITD_Research_Intern/models/research:/content/models/research/slim

In [0]:
!protoc object_detection/protos/*.proto --python_out=.

In [0]:
!python object_detection/builders/model_builder_test.py

In [0]:
from tqdm import tqdm
for image_path in tqdm(TEST_IMAGE_PATHS):
  print(image_path[12:])
  image = Image.open(image_path)
  # the array based representation of the image will be used later in order to prepare the
  # result image with boxes and labels on it.
  image_np = load_image_into_numpy_array(image)
  # Expand dimensions since the model expects images to have shape: [1, None, None, 3]
  image_np_expanded = np.expand_dims(image_np, axis=0)
  # Actual detection.
  output_dict = run_inference_for_single_image(image_np_expanded, detection_graph)
  # Visualization of the results of a detection.
  vis_util.visualize_boxes_and_labels_on_image_array(
      image_np,
      output_dict['detection_boxes'],
      output_dict['detection_classes'],
      output_dict['detection_scores'],
      category_index,
      instance_masks=output_dict.get('detection_masks'),
      use_normalized_coordinates=True,
      line_thickness=8)
  plt.figure(figsize=IMAGE_SIZE)
  #plt.imshow(image_np)